In [1]:
!pip install transformers torch datasets indic-nlp-library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 10.7 MB/s eta 0:00:00


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    predicted_class = torch.argmax(probs).item()

    # Mapping (model gives 1–5 stars)
    if predicted_class <= 1:
        return "Negative"
    elif predicted_class == 2:
        return "Neutral"
    else:
        return "Positive"

In [4]:
texts = [
    "यह फिल्म बहुत अच्छी है",        # Hindi (Positive)
    "हा चित्रपट खूप वाईट आहे",      # Marathi (Negative)
    "This movie is okay",            # English (Neutral)
    "movie bahut mast hai",          # Hinglish
]

for text in texts:
    print(f"Text: {text}")
    print("Sentiment:", predict_sentiment(text))
    print("-" * 40)

Text: यह फिल्म बहुत अच्छी है
Sentiment: Positive
----------------------------------------
Text: हा चित्रपट खूप वाईट आहे
Sentiment: Negative
----------------------------------------
Text: This movie is okay
Sentiment: Neutral
----------------------------------------
Text: movie bahut mast hai
Sentiment: Negative
----------------------------------------


In [5]:
user_text = input("Enter text in any Indian language: ")
print("Sentiment:", predict_sentiment(user_text))

Enter text in any Indian language: Mujhe bahar jaana hai par baarish ho rahi hai
Sentiment: Negative


In [7]:
def predict_with_confidence(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    print("Probabilities:", probs.detach().numpy())
    print("Predicted Class:", torch.argmax(probs).item())